In [13]:
import pandas as pd
import numpy as np
import sys 
import os


In [14]:
import pickle

SIMILARITY_METHOD = "kendall"
WINDOW_SIZE = 7
STEP_SIZE = 1
# Caminho para o ficheiro exportado (ajuste se guardou noutra pasta)
pkl_path = f'../GraphAnalysis/{SIMILARITY_METHOD}/dynamic_graphs_output_{SIMILARITY_METHOD}_0.8_Window{WINDOW_SIZE}_Step{STEP_SIZE}.pkl'

print(f"A carregar grafos e features pré-computados de {pkl_path}...")
with open(pkl_path, 'rb') as f:
    graph_data = pickle.load(f)

# Extrair todas as variáveis de volta para o ambiente do notebook
graphs = graph_data['graphs']
sim_dfs = graph_data['sim_dfs']
df_pivots = graph_data['df_pivots']
window_info = graph_data['window_info']
dynamic_graph_features = graph_data['dynamic_graph_features']



print(f"Sucesso! Foram carregadas as informações de {len(graphs)} janelas temporais.")


A carregar grafos e features pré-computados de ../GraphAnalysis/kendall/dynamic_graphs_output_kendall_0.8_Window7_Step1.pkl...
Sucesso! Foram carregadas as informações de 603 janelas temporais.


In [15]:
'''
from DynamicSimilarities.plot import save_dynamic_graph_plots
SIMILARITY_METHOD = 'kendall'
WINDOW_SIZE = 7
STEP_SIZE = 1
save_dynamic_graph_plots(graphs, SIMILARITY_METHOD, node_categories=None, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
'''

"\nfrom DynamicSimilarities.plot import save_dynamic_graph_plots\nSIMILARITY_METHOD = 'kendall'\nWINDOW_SIZE = 7\nSTEP_SIZE = 1\nsave_dynamic_graph_plots(graphs, SIMILARITY_METHOD, node_categories=None, window_size=WINDOW_SIZE, step_size=STEP_SIZE)\n"

In [16]:
import networkx as nx
from collections import Counter
sys.path.append(os.path.abspath('../..'))
from DynamicSimilarities.plot import save_graph_plot

# 1. Count how many times each edge appears across all temporal graphs
edge_counts = Counter()
total_graphs = len(graphs)

for g in graphs:
    # Ensure edge tuples are sorted so (A, B) and (B, A) are counted as the same edge
    for u, v in g.edges():
        edge = tuple(sorted((str(u), str(v))))
        edge_counts[edge] += 1

# 2. Create a DataFrame to analyze the persistence of each relationship
persistent_edges = pd.DataFrame(
    [(u, v, count, count / total_graphs * 100) for (u, v), count in edge_counts.items()],
    columns=['Node1', 'Node2', 'Occurrences', 'Persistence (%)']
)

# Sort by persistence (descending) to find the most stable relationships
persistent_edges = persistent_edges.sort_values(by='Persistence (%)', ascending=False).reset_index(drop=True)

print(f"Analysis over {total_graphs} temporal graphs.")
print(f"Total unique edges that appeared at least once: {len(persistent_edges)}")
print("\nTop 20 most stable product relationships:")
display(persistent_edges.head(20))

# 3. Optional: Create a "Consensus Graph" with edges present in at least X% of the time (e.g., 50%)
THRESHOLD_PERCENT = 50.0
stable_graph = nx.Graph()

# Add only the edges that meet the threshold
stable_edges = persistent_edges[persistent_edges['Persistence (%)'] >= THRESHOLD_PERCENT]
for _, row in stable_edges.iterrows():
    stable_graph.add_edge(row['Node1'], row['Node2'], weight=row['Persistence (%)'])

print(f"\nCreated a Consensus Graph with {stable_graph.number_of_edges()} edges (Persistence >= {THRESHOLD_PERCENT}%)")

# Optional: Save the consensus graph plot
save_graph_plot(
    G=stable_graph, 
    strategy_name=f"Consensus_Graph_Persistence_{SIMILARITY_METHOD}_{int(THRESHOLD_PERCENT)}",
    output_folder="consensus_graph_plots",
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE
)

Analysis over 603 temporal graphs.
Total unique edges that appeared at least once: 967460

Top 20 most stable product relationships:


,Node1,Node2,Occurrences,Persistence (%)
0,907969,911753,107,17.744610
1,26008,911753,75,12.437811
2,213627,907969,66,10.945274
3,26008,907969,65,10.779436
4,213628,907969,60,9.950249
5,213629,907969,58,9.618574
6,213627,911753,58,9.618574
7,26008,932812,58,9.618574
8,110187,907969,58,9.618574
9,213625,213628,55,9.121061



Created a Consensus Graph with 0 edges (Persistence >= 50.0%)
Graph automatically saved to: consensus_graph_plots\consensus_graph_persistence_kendall_50_0_nodes.png


In [23]:
import numpy as np
import pandas as pd
import networkx as nx

# 1. Collect histories for degree, betweenness, and eigenvector centrality
node_degree_history = {}
node_betweenness_history = {}
node_eigenvector_history = {}

print("Calculating degree and centrality metrics across all windows. This may take a moment...")
for w_idx, g in enumerate(graphs):
    # Calculate centralities for the current window
    # Weight handling: if your edges have weights, networkx uses them differently.
    # For betweenness, nx expects "distance" or cost, so we often ignore it or invert it. Let's ignore it for basic structural betweenness.
    # For eigenvector, weight represents connection strength, which is good.
    betweenness = nx.betweenness_centrality(g, weight=None)
    
    try:
        # Eigenvector centrality can sometimes fail to converge, so we use a try-except block
        eigenvector = nx.eigenvector_centrality(g, max_iter=500, weight='weight')
    except nx.PowerIterationFailedConvergence:
        eigenvector = {n: 0 for n in g.nodes()} # Fallback if it fails

    for node in g.nodes():
        # Initialize histories if node not seen yet
        if node not in node_degree_history:
            node_degree_history[node] = []
            node_betweenness_history[node] = []
            node_eigenvector_history[node] = []
            
        # Append metrics
        node_degree_history[node].append(g.degree[node])
        node_betweenness_history[node].append(betweenness.get(node, 0))
        node_eigenvector_history[node].append(eigenvector.get(node, 0))

# 2. Compute statistics across time for all metrics
node_stats = []
for node in node_degree_history.keys():
    degrees = node_degree_history[node]
    betweenness_vals = node_betweenness_history[node]
    eigenvector_vals = node_eigenvector_history[node]
    
    node_stats.append({
        'Node': node,
        # Degree stats
        'Mean_Degree': np.mean(degrees),
        'Median_Degree': np.median(degrees),
        'Std_Degree': np.std(degrees),
        'Max_Degree': np.max(degrees),
        'Zero_Degree_Windows': sum(1 for d in degrees if d == 0),
        
        # Centrality stats (using Mean as the primary aggregator)
        'Mean_Betweenness': np.mean(betweenness_vals),
        'Mean_Eigenvector': np.mean(eigenvector_vals)
    })

df_node_stats = pd.DataFrame(node_stats)

# 3. Create a combined Stability Score.
# We want nodes that are consistently connected (high mean degree, low std dev).
# We also might want nodes that are influential in their local clusters (Eigenvector)
# or act as bridges between different product types (Betweenness).
df_node_stats['Degree_Stability'] = df_node_stats['Mean_Degree'] / (df_node_stats['Std_Degree'] + 1e-5)

# Example: Sort by Eigenvector Centrality to find the most "influential" nodes in the network
df_node_stats = df_node_stats.sort_values(by='Mean_Eigenvector', ascending=False).reset_index(drop=True)

print("Calculation complete!")
display(df_node_stats.head())


Calculating degree and centrality metrics across all windows. This may take a moment...
Calculation complete!


,Node,Mean_Degree,Median_Degree,Std_Degree,Max_Degree,Zero_Degree_Windows,Mean_Betweenness,Mean_Eigenvector,Degree_Stability
0,907969,22.326700,20.0,13.721020,129,0,0.001469,0.049036,1.627188
1,911753,21.610282,19.0,13.207580,129,0,0.001473,0.045559,1.636202
2,213629,19.898839,17.0,13.449668,133,0,0.001714,0.035539,1.479503
3,110187,18.189055,16.0,13.965514,132,0,0.001287,0.034912,1.302426
4,26008,19.582090,16.0,14.372830,133,0,0.001319,0.034461,1.362437


In [ ]:
df_node_stats.to_csv(f'node_stats_{SIMILARITY_METHOD}_Window{WINDOW_SIZE}_Step{STEP_SIZE}.csv', index=False)

In [20]:

well_connected_nodes = df_node_stats[
    (df_node_stats['Median_Degree'] >= 5) & 
    (df_node_stats['Zero_Degree_Windows'] == 0) 
].sort_values(by=['Stability_Score', 'Mean_Degree'], ascending=[False, False])

print(f"Found {len(well_connected_nodes)} nodes that meet the criteria.")
print("\nTop 20 nodes for Node2Vec/LSTM testing:")
display(well_connected_nodes.head(20))

# Save the list of good candidate nodes for later use
candidate_nodes = well_connected_nodes['Node'].tolist()

Found 50 nodes that meet the criteria.

Top 20 nodes for Node2Vec/LSTM testing:


,Node,Mean_Degree,Median_Degree,Std_Degree,Min_Degree,Max_Degree,Zero_Degree_Windows,Stability_Score
1423,526768,12.632653,12.0,6.814977,1,44,0,1.853658
1090,298216,13.912106,12.0,8.409734,1,75,0,1.654284
1322,911753,21.610282,19.0,13.207580,3,129,0,1.636202
1318,907969,22.326700,20.0,13.721020,3,129,0,1.627188
210,13245,13.321725,12.0,8.217361,1,43,0,1.621166
927,213448,16.034826,14.0,10.078487,1,71,0,1.590994
380,41718,10.782753,9.0,6.857292,1,43,0,1.572448
1155,538923,17.771144,16.0,11.426484,2,95,0,1.555258
83,1765,13.961857,12.0,9.049351,1,54,0,1.542856
291,21178,15.330017,13.0,9.946067,1,65,0,1.541313


In [ ]:
lowest_max_degree = df_node_stats['Max_Degree'].min()


In [24]:
import networkx as nx
from networkx.algorithms.community import louvain_communities
from collections import defaultdict

# 1. Detect communities in each individual window using Louvain
window_communities = []

print("Detecting communities in each time window...")
for idx, g in enumerate(graphs):
    # Louvain returns a list of sets, where each set is a community of nodes
    try:
        # If your edges have weights, Louvain can use them to find better clusters
        comms = louvain_communities(g, weight='weight') 
    except:
        # Fallback if weight causes an issue
        comms = louvain_communities(g)
    
    window_communities.append(comms)

# 2. Track "co-occurrence" of nodes in the same community across time
# We will build a matrix counting how many times node A and node B ended up in the SAME cluster.
node_list = list(graphs[0].nodes()) # Assuming all graphs have the same nodes
co_occurrence_counts = defaultdict(int)

total_windows = len(window_communities)

for comms in window_communities:
    for community in comms:
        # For every pair of nodes in the same community, increment their co-occurrence score
        nodes_in_comm = list(community)
        for i in range(len(nodes_in_comm)):
            for j in range(i + 1, len(nodes_in_comm)):
                u, v = nodes_in_comm[i], nodes_in_comm[j]
                # Ensure consistent ordering so (A,B) and (B,A) map to the same key
                pair = tuple(sorted([str(u), str(v)]))
                co_occurrence_counts[pair] += 1

# 3. Filter for pairs that are clustered together in at least X% of the windows
STABLE_CLUSTER_THRESHOLD = 0.50 # e.g., clustered together 50% of the time combined
min_co_occurrences = total_windows * STABLE_CLUSTER_THRESHOLD

stable_pairs = {pair: count for pair, count in co_occurrence_counts.items() if count >= min_co_occurrences}

print(f"\nFound {len(stable_pairs)} product pairs that belong to the SAME cluster in at least {STABLE_CLUSTER_THRESHOLD*100}% of the windows.")

# 4. Build a "Stable Community Graph" to extract the final robust clusters
stable_cluster_graph = nx.Graph()
for (u, v), count in stable_pairs.items():
    stable_cluster_graph.add_edge(u, v, weight=count)

# Extract connected components from this stable graph. 
# These components represent your stable, long-lasting clusters.
stable_components = list(nx.connected_components(stable_cluster_graph))
stable_components.sort(key=len, reverse=True) # Sort by size

print(f"\nExtracted {len(stable_components)} stable clusters over time.")
for i, comp in enumerate(stable_components[:5]): # Print top 5 largest stable clusters
    print(f"Stable Cluster {i+1} (Size {len(comp)}): {list(comp)[:10]}...")

Detecting communities in each time window...


KeyboardInterrupt: 